# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [2]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [ ]:
#: Import the necessary libs
# For example: 
import os
import chromadb
from chromadb.utils import embedding_functions
from typing import Annotated

from datetime import datetime
from typing import List, Dict, TypedDict
from lib.agents import Agent, AgentState as State
from tavily import TavilyClient
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage, BaseMessage
from lib.tooling import tool
from lib.vector_db import VectorStoreManager, CorpusLoaderService
from lib.rag import RAG
from lib.evaluation import AgentEvaluator
from lib.state_machine import StateMachine, Step, EntryPoint, Termination, Resource
from lib.parsers import PydanticOutputParser
from pydantic import BaseModel, Field

In [ ]:
#: Load environment variables
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [5]:
db = VectorStoreManager(OPENAI_API_KEY)


In [6]:
loader_service = CorpusLoaderService(db)

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [9]:
# : Create retrieve_game tool
# It should use chroma client and collection you created

# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

chroma_client= chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection("udaplay")

resource = Resource(vars={"collection": collection})

@tool
def retrieve_game(query: str):
    collection:Collection = resource.vars.get("collection")

    results = collection.query(
        query_texts = [query],
        n_results=3,
        include=['metadatas']
    )
     
    metadatas = results.get('metadatas', [[]])[0] 
    
    result_list =[]

    for metadata in metadatas:
      result_list.append({
        "Platform": metadata.get("Platform"),
        "Name": metadata.get("Name"),
        "YearOfRelease": metadata.get("YearOfRelease"),
        "Description": metadata.get("Description")
    }
      )
    return result_list


#### Evaluate Retrieval Tool

In [10]:
# : Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result
class EvaluationReport(BaseModel):
      useful: Annotated[bool, Field(description="whether the documents are useful to answer the question", default=None)]
      description: Annotated[str, Field(description="description about the evaluation result", default=None)]
    
@tool
def evaluate_retrieval(question, retrieved_docs):
    rag_llm = LLM(
    model="gpt-4o-mini",
    temperature=0.3)

    prompt = f"""
            Your task is to evaluate if the documents are enough to respond the query.
            Give a detailed explanation, so it's possible to take an action to accept it or not.

            Here's the query: {question}
            Here's the retrieved_docs: {retrieved_docs}

            provide the explanation in detailed format as {EvaluationReport}
    """

    result = rag_llm.invoke(
      input=prompt,
      response_format=EvaluationReport
    )
     
    parser = PydanticOutputParser(model_class=EvaluationReport)
    

    return parser.parse(result)
    



#### Game Web Search Tool

In [11]:
# : Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

@tool
def game_web_search(query):
    api_key = os.getenv("TAVILY_API_KEY")
    client = TavilyClient(api_key=api_key)

    search_result = client.search(
        query = query,
        include_answer=True,
        include_raw_content=False,
        include_images=False
    )

    final_result = {
        "answer" : search_result.get("answer", ""),
        "results" : search_result.get('results',[]),
        "search_metadata" : {
            "timestamp": datetime.now().isoformat(),
            "query": query
        }
    }
    return final_result


### Agent

In [46]:
# : Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed
agentic_rag = Agent(
    model_name="gpt-4o-mini",
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    instructions=(
        """ You are an Agentic assistant specialized in gaming analytics. Your role is to answer the question by following the exact steps mentioned below.
        1. First attempt to answer using internal knowledge by using the {retrieve_game} tool.
        2. Next, use the retrieved information from step 1 to evaluate the result using {evaluate_retrieval} tool.
        3. if you did not find correct answer, then fall back to web search using the {game_web_search} tool.
        
        The output after each tool call should include the format exactly as below:
         Tool: <tool name>

         Result from tool: <result>

         Reasoning from the result: <reasoning>

         ----------------------------------------------
         
         Final answer: include the final answer here if no more tools have to be called. Include the citations of sources you got the answer
        Always explain your answer and reasoning after calling each tool selection and provide concise answers.
        if the final correct answer is found, stop
        """
    )
)

In [48]:
# Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?

def print_messages(messages: List[BaseMessage]):
    for m in messages:
        print(f" -> (role = {m.role}, content = {m.content}, tool_calls = {getattr(m, 'tool_calls', None)})")

questions = ['When Pokémon Gold and Silver was released?', 'Which one was the first 3D platformer Mario game?', 'Was Mortal Kombat X realeased for Playstation 5?' ]

session = "project_3"
for q in questions:
    run_obj = agentic_rag.invoke(query=q, session_id=session)
    messages = run_obj.get_final_state()["messages"]
    print(f"\n=== Question: {q} ===")

    #print_messages(messages)
    print(run_obj.get_final_state()["messages"][-1].content)

    print(f"\n=====================")

    

    


[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

=== Question: When Pokémon Gold and Silver was released? ===
Tool: functions.retrieve_game

Result from tool: [{'Platform': 'Game Boy Color', 'Name': 'Pokémon Gold and Silver', 'YearOfRelease': 1999, 'Description': 'Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.'}, {'Platform': 'Game Boy Advance', 'Name': 'Pokémon Ruby and Sapphire', 'YearOfRelease': 2002, 'Description': 'Third-generation Pokémon games set in the Hoenn region, featuring new Pokémon and double battles.'}, {'Platform': 'Nintendo Switch', 'Name': 'Mario Kart 8 Deluxe', 'YearOfRelease': 2017, 'Description': 'An enhanced version of Mario K

### (Optional) Advanced

In [ ]:
# : Update your agent with long-term memory
# : Convert the agent to be a state machine, with the tools being pre-defined nodes